# 🗓️ 8일차 스터디 노트북 — 해시법 & 체인법

**오늘 범위**: 03-4 해시법 → 해시값·해시 테이블·버킷 → 해시 충돌 → 체인법(Node/ChainedHash) → add/search/remove/dump

## 난이도 태그
- 🟢 **기본** — 전원 필수 (개념을 쓸 수 있는지)
- 🟡 **표준** — 팀 목표선 (왜 그런지 + 변형/디버깅)
- 🔴 **심화** — 도전 (효율·엣지케이스·일반화)

## 유형 태그
[예측] 실행 전 결과 맞히기 · [구현] TODO 채우기 · [디버깅] 버그 찾기 · [설명] 왜인지 서술 · [실험] 직접 찍어보기

## 푸는 법
1. 위에서부터 순서대로 (Remind → 본문 🟢 → 🟡 → 🔴)
2. 코드 셀은 직접 실행해서 확인
3. 막히면 힌트만 보고, 정답은 맨 아래에서
4. 금요일 코딩테스트는 🟢🟡 범위

> 📁 이 노트북의 코드 셀은 `chained_hash.py`가 **같은 폴더**에 있다고 가정해. 없으면 아래 "부록: ChainedHash 전체 코드" 셀을 먼저 실행하면 돼.

---

## 🔁 [Remind] 워밍업 — 7일차 복습 (한 단계 업)

7일차 이진 검색 & 복잡도를 해시법으로 넘어가기 전에 다시 짚자.

### R-1. 🟡 [설명] 이진검색은 빠른데, 왜 해시법이 필요해?

7일차에 이진 검색이 O(log n)으로 엄청 빠르다는 걸 봤어 (n=100000일 때 17번). 그런데 교재 131p는 **정렬된 배열에 원소를 추가**하는 얘기부터 시작해.

- 정렬된 배열 `[5, 6, 14, 20, 29, 34, 37, 51, 69, 75]`에 `35`를 추가하려면 어떤 작업이 필요하고, 그 복잡도는 얼마야?
- "검색은 O(log n)으로 빠른데" 왜 이게 문제가 되지? 교재가 해시법을 꺼내는 **동기**를 설명해봐.

*(힌트: 교재 110p에 "검색뿐 아니라 데이터의 추가·삭제도 자주 수행해야 한다면 검색 이외의 작업에 들어가는 비용을 종합 평가하여 알고리즘을 선택해야 합니다"라고 나와 있어.)*

*(여기에 답 작성)*

### R-2. 🔴 [디버깅] 이진검색 종료 조건, 또 틀렸다

아래 이진검색은 종료 조건이 틀렸어. 어디가 왜 틀렸는지 찾고, 실행해서 확인해봐.
(7일차에 우리가 **세 번** 헷갈린 그 지점이야.)

In [ ]:
from typing import Any, Sequence

def bin_search(a: Sequence, key: Any) -> int:
    pl, pr = 0, len(a) - 1
    while True:
        pc = (pl + pr) // 2
        if a[pc] == key:
            return pc
        elif a[pc] < key:
            pl = pc + 1
        else:
            pr = pc - 1
        if pr < pc:   # <- 여기가 문제
            break
    return -1

a = [1, 3, 5, 7, 9, 11, 13]
for k in [1, 3, 5, 7, 9, 11, 13]:
    print(k, '->', bin_search(a, k), '(정답:', a.index(k), ')')


---
## 📦 부록 — ChainedHash 전체 코드

`chained_hash.py`가 같은 폴더에 없다면 **이 셀을 먼저 실행**해. (있으면 건너뛰어도 됨)

In [ ]:
from __future__ import annotations
from typing import Any
import hashlib

class Node:
    '''해시를 구성하는 노드'''
    def __init__(self, key: Any, value: Any, next: Node) -> None:
        self.key = key
        self.value = value
        self.next = next

class ChainedHash:
    '''체인법으로 해시 클래스 구현'''
    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.table = [None] * self.capacity

    def hash_value(self, key: Any) -> int:
        '''해시값을 구함'''
        if isinstance(key, int):
            return key % self.capacity
        return (int(hashlib.sha256(str(key).encode()).hexdigest(), 16) % self.capacity)

    def search(self, key: Any) -> Any:
        '''키가 key인 원소를 검색하여 값을 반환'''
        hash = self.hash_value(key)
        p = self.table[hash]
        while p is not None:
            if p.key == key:
                return p.value
            p = p.next
        return None

    def add(self, key: Any, value: Any) -> bool:
        '''키가 key이고 값이 value인 원소를 추가'''
        hash = self.hash_value(key)
        p = self.table[hash]
        while p is not None:
            if p.key == key:
                return False
            p = p.next
        temp = Node(key, value, self.table[hash])
        self.table[hash] = temp
        return True

    def remove(self, key: Any) -> bool:
        '''키가 key인 원소를 삭제'''
        hash = self.hash_value(key)
        p = self.table[hash]
        pp = None
        while p is not None:
            if p.key == key:
                if pp is None:
                    self.table[hash] = p.next
                else:
                    pp.next = p.next
                return True
            pp = p
            p = p.next
        return False

    def dump(self) -> None:
        '''해시 테이블을 덤프'''
        for i in range(self.capacity):
            p = self.table[i]
            print(i, end='')
            while p is not None:
                print(f'  -> {p.key} ({p.value})', end='')
                p = p.next
            print()

print('ChainedHash 준비 완료')


---
## 📖 오늘의 핵심 개념

### 해시법(hashing)
**"데이터를 저장할 위치 = 인덱스"를 간단한 연산으로 구하는 것.**

- **해시 함수**: 키 → 해시값으로 바꾸는 함수. 보통 **나머지 연산(`%`)**
- **해시값**: `key % capacity`. 데이터에 접근할 때의 기준
- **해시 테이블**: 해시값을 인덱스로 삼아 원소를 저장하는 배열
- **버킷(bucket)**: 해시 테이블의 각 원소(칸)

정렬 배열은 원소 추가 시 뒤쪽을 전부 밀어야 해서 **O(n)**. 해시법은 계산으로 위치를 바로 구하니까 **밀 필요가 없음** → 추가·삭제도 효율적.

### 해시 충돌(collision)
키와 해시값의 대응은 1:1이 아니라 **n:1**. (13으로 나눈 나머지가 같은 키는 여러 개 → 예: 1, 14, 27...)
저장할 버킷이 중복되는 현상이 **충돌**.

> 해시 함수는 해시값이 한쪽으로 치우치지 않고 **고르게 분산**되도록 만드는 게 가장 좋음.

**충돌 대처법 2가지**:
- **체인법(chaining, = 오픈 해싱)**: 해시값이 같은 원소를 **연결 리스트**로 관리 ← 오늘 배운 것
- **오픈 주소법(open addressing)**: 빈 버킷을 찾을 때까지 해시를 반복

### 체인법 구조
- **Node**: `key`, `value`, `next`(뒤쪽 노드 참조) — **자기 참조형** 클래스
- **ChainedHash**: `capacity`(테이블 크기), `table`(list형 배열)
- 각 버킷은 "해시값이 같은 노드를 연결한 리스트"의 **앞쪽 노드(head node)**를 참조

### 3대 연산 (전부 같은 패턴!)
1. 해시 함수로 키를 해시값으로 변환
2. 해시값을 인덱스로 하는 버킷에 주목
3. 버킷이 참조하는 **연결 리스트를 맨 앞부터 선형 검색** ← 6일차 선형 검색이 여기서 재등장!

| 연산 | 찾으면 | 못 찾으면 |
|---|---|---|
| `search` | `p.value` 반환 | `None` 반환 |
| `add` | `False`(이미 등록됨) | 리스트 **맨 앞**에 노드 추가, `True` |
| `remove` | 리스트에서 노드 제거, `True` | `False` |

### 복잡도 & 트레이드오프
- **충돌이 전혀 없다면** 검색·추가·삭제 모두 **O(1)**
- 해시 테이블을 충분히 크게 만들면 충돌은 억제되지만 **메모리를 낭비** → **시간과 공간의 트레이드오프**
- 해시 테이블 크기는 **소수(prime)**를 선호

---
## 🟢 기본 문제

### 1. [예측] 해시값 계산 & dump 결과 맞히기

`ChainedHash(13)`에 아래 순서대로 추가한다고 하자. **실행 전에** 각 키의 해시값과 최종 dump 결과를 손으로 계산해봐.

| 키 | 해시값(13으로 나눈 나머지) |
|---|---|
| 1 | ? |
| 5 | ? |
| 10 | ? |
| 12 | ? |
| 14 | ? |

- 충돌이 일어나는 키 쌍은 뭐야?
- 버킷 1에는 어떤 순서로 노드가 연결될까? (`14 → 1`? `1 → 14`?)

In [ ]:
# 예측 먼저 하고 실행!
h = ChainedHash(13)
h.add(1, '수연')
h.add(5, '동혁')
h.add(10, '예지')
h.add(12, '원준')
h.add(14, '민서')
h.dump()


### 2. [구현] hash_value 채우기

`hash_value`의 TODO를 채워봐. int일 때와 아닐 때를 나눠서 처리해야 해.

In [ ]:
import hashlib
from typing import Any

class MyHash:
    def __init__(self, capacity: int) -> None:
        self.capacity = capacity

    def hash_value(self, key: Any) -> int:
        '''해시값을 구함'''
        if isinstance(key, int):
            return ___   # TODO: int면 어떻게?
        # TODO: int가 아니면 sha256 -> hexdigest -> int(16진수) -> % capacity
        return ___

m = MyHash(13)
print(m.hash_value(14))       # 1
print(m.hash_value(35))       # 9
print(type(m.hash_value('강아지')))  # <class 'int'> 여야 함


### 3. [설명] 충돌이란 무엇인가

- 키와 해시값의 대응 관계가 **1:1이 아니라 n:1**인 이유를 설명해봐.
- `ChainedHash(13)`에서 키 `18`을 추가하려는데 버킷 5에 이미 값이 있다면, 이걸 뭐라고 부르고 체인법은 어떻게 대처해?

*(여기에 답 작성)*

---
## 🟡 표준 문제

### 4. [디버깅] 내장 이름 가리기 🔥🔥

`chained_hash_test.py` 17행에 이런 코드가 있어 (교재 142p도 동일):

```python
hash = ChainedHash(13)
```

그리고 `chained_hash.py`의 `search`/`add`/`remove` 안에도 이런 게 있어:

```python
hash = self.hash_value(key)
```

**이게 왜 위험한 패턴인지** 아래 셀을 실행해서 직접 확인해봐. 그리고 어떻게 고치면 좋을지 제안해봐.

In [ ]:
# 실행 전 예측: 마지막 print(hash('abc'))는 어떻게 될까?

print('원래 hash:', hash)          # 내장 함수
print('hash("abc") =', hash('abc'))  # 잘 됨

hash = ChainedHash(13)   # <- 교재/우리 코드가 하는 일
print('덮어쓴 뒤 hash:', hash)

# 이제 내장 hash를 다시 쓰려고 하면?
try:
    print(hash('abc'))
except TypeError as e:
    print('TypeError 발생:', e)


### 5. [예측] add는 왜 "맨 앞"에 추가할까?

`add`의 핵심 두 줄이야:

```python
temp = Node(key, value, self.table[hash])
self.table[hash] = temp
```

- `Node(key, value, self.table[hash])`에서 세 번째 인자로 **기존 head**를 넘기는 게 무슨 의미야?
- 만약 "맨 뒤"에 추가하도록 바꾸면 복잡도가 어떻게 달라질까?
- 아래 실행 결과에서 dump 순서가 추가 순서의 **역순**인 이유를 설명해봐.

In [ ]:
h = ChainedHash(13)
h.add(1, 'A')    # 해시값 1
h.add(14, 'B')   # 해시값 1 (충돌!)
h.add(27, 'C')   # 해시값 1 (충돌!)
h.dump()
# 버킷 1의 순서는? 추가 순서(1->14->27)? 역순(27->14->1)?


### 6. [디버깅] remove에서 `pp is None` 케이스를 빼먹으면?

아래는 `remove`에서 `pp is None` 분기를 빼먹은 버전이야. 어떤 상황에서 터질지 예측하고 실행해서 확인해봐.

*(힌트: `pp`는 "바로 앞의 노드"를 추적하는 변수야. 그럼 삭제 대상이 **리스트의 첫 번째 노드**일 땐 `pp`가 뭐지?)*

In [ ]:
class BuggyHash(ChainedHash):
    def remove(self, key):
        hash = self.hash_value(key)
        p = self.table[hash]
        pp = None
        while p is not None:
            if p.key == key:
                pp.next = p.next   # <- pp is None 체크가 없음!
                return True
            pp = p
            p = p.next
        return False

h = BuggyHash(13)
h.add(1, 'A')
h.add(14, 'B')   # 버킷 1: 14 -> 1

print('중간/끝 노드 삭제 (1):', end=' ')
try:
    print(h.remove(1))
except AttributeError as e:
    print('AttributeError:', e)

print('맨 앞 노드 삭제 (14):', end=' ')
try:
    print(h.remove(14))
except AttributeError as e:
    print('AttributeError:', e)


---
## 🔴 심화 문제

### 7. [실험] 해시 테이블 크기, 왜 소수가 좋을까?

교재 136p: "해시 테이블의 크기는 소수를 선호합니다."

같은 데이터를 **소수 크기(13)** 테이블과 **합성수 크기(12)** 테이블에 넣어보고, 충돌이 얼마나 다르게 분포하는지 실험해봐.

In [ ]:
def collision_stats(capacity, keys):
    '''각 버킷에 몇 개씩 들어가는지 세기'''
    buckets = [0] * capacity
    for k in keys:
        buckets[k % capacity] += 1
    used = sum(1 for b in buckets if b > 0)
    max_chain = max(buckets)
    return buckets, used, max_chain

# 짝수 배수가 많은 데이터셋 (실무에서 흔함: ID가 특정 간격으로 증가)
keys = [i * 6 for i in range(1, 21)]   # 6, 12, 18, ... 120
print('키:', keys)
print()

for cap in [12, 13]:
    buckets, used, max_chain = collision_stats(cap, keys)
    kind = '소수' if cap == 13 else '합성수'
    print(f'capacity={cap} ({kind})')
    print(f'  버킷 분포: {buckets}')
    print(f'  사용된 버킷 수: {used}/{cap}')
    print(f'  가장 긴 체인: {max_chain}')
    print()


### 8. [구현] 체인 길이를 재는 메서드 추가

`ChainedHash`에 아래 두 메서드를 추가해봐.

- `chain_length(key)`: 그 키가 속한 버킷의 체인 길이를 반환
- `load_factor()`: 적재율 = (전체 원소 수) / capacity 를 반환

이게 왜 유용하냐면, 적재율이 높아지면 체인이 길어져서 **O(1)이 O(n)으로 퇴화**하거든. 실무 해시맵은 이 값을 보고 테이블을 키워(rehashing).

In [ ]:
class StatsHash(ChainedHash):
    def chain_length(self, key) -> int:
        '''key가 속한 버킷의 체인 길이'''
        # TODO
        pass

    def load_factor(self) -> float:
        '''전체 원소 수 / capacity'''
        # TODO: 모든 버킷을 순회하며 노드 개수를 세야 해
        pass

h = StatsHash(13)
for k in [1, 14, 27, 5]:
    h.add(k, f'v{k}')

print(h.chain_length(1))    # 3  (1, 14, 27이 버킷 1에 모임)
print(h.chain_length(5))    # 1
print(h.chain_length(2))    # 0  (빈 버킷)
print(round(h.load_factor(), 4))  # 4/13 = 0.3077


### 9. [설명] 어노테이션 & 트레이드오프

**(a)** 우리 `chained_hash.py`에는 이렇게 되어 있어:

```python
def __init__(self, capacity: int) -> int:   # 우리 코드
def __init__(self, capacity: int) -> None:  # 교재 135p
```

`-> int`인데도 에러 없이 잘 돌아가는 이유가 뭘까? (4일차 함수 어노테이션 복습) 그럼 어노테이션은 왜 쓰는 걸까?

**(b)** 교재 136p: "해시 테이블을 충분히 크게 만들면 충돌 발생을 억제할 수 있지만 이 방법은 메모리를 낭비합니다. 즉, 시간과 공간의 트레이드-오프 문제가 발생합니다."

- capacity를 무한히 크게 하면 어떤 이득과 손해가 있어?
- capacity를 1로 하면 ChainedHash는 사실상 어떤 자료구조가 될까? 그때 검색 복잡도는?

*(여기에 답 작성)*

---
## ✅ 정답 & 해설

### R-1. 이진검색이 있는데 왜 해시법?

**정렬 배열에 35 추가하기** (교재 131p):
1. `x[5]`와 `x[6]` 사이에 들어가야 함을 **이진 검색**으로 찾음 → O(log n) ✓ 빠름
2. `x[6]` 이후의 **모든 원소를 한 칸씩 뒤로 이동** → **O(n)** ✗ 비쌈
3. `x[6]`에 35 대입

**왜 문제인가**: 검색 자체는 O(log n)으로 빠르지만, **"어디에 넣을지 찾는 비용"이 아니라 "넣을 자리를 만드는 비용"**이 O(n)이야. 삭제도 마찬가지(뒤 원소를 앞으로 당겨야 함). 그래서 검색만 많이 하는 상황이면 이진 검색으로 충분하지만, **추가·삭제가 자주 일어나면** 전체 비용이 O(n)에 발목 잡혀.

해시법의 동기: **위치를 계산으로 바로 구하니까 원소를 밀 필요가 없다** → 추가·삭제도 O(1)에 가깝게. 이게 "검색만 잘되면 좋지!"가 아니라 "종합 평가"를 해야 하는 이유(교재 110p).

---
### R-2. 이진검색 종료 조건 버그

**버그**: `if pr < pc: break`

**왜 틀렸나**: `else` 분기(`pr = pc - 1`)를 타면 `pr`이 방금 `pc - 1`로 대입됐으니 `pr < pc`는 **항상 참** → 왼쪽 절반이 남았는데도 즉시 종료. 7일차 버전 A(`pr == pc - 1`)와 사실상 같은 버그야.

**해결**: `if pl > pr: break`

**교훈(또!)**: 종료 조건은 탐색 경계 **`pl`과 `pr`끼리** 비교. 방금 본 위치(`pc`)를 끌어들이면 분기에 따라 우연히 참이 돼서 조기 종료돼.

---
### 1. 해시값 & dump 예측

| 키 | 해시값 |
|---|---|
| 1 | 1 |
| 5 | 5 |
| 10 | 10 |
| 12 | 12 |
| 14 | **1** ← 충돌! |

- **충돌 쌍**: 1과 14 (둘 다 13으로 나눈 나머지가 1)
- **버킷 1의 순서**: `14 → 1` (나중에 추가한 14가 **맨 앞**). `add`가 리스트 맨 앞에 삽입하기 때문 (5번 문제 참고)
- 교재 143p 실행결과 `1 → 14 (민서) → 1 (수연)`과 일치

---
### 2. hash_value 구현 정답

In [ ]:
import hashlib
from typing import Any

class MyHash:
    def __init__(self, capacity: int) -> None:
        self.capacity = capacity

    def hash_value(self, key: Any) -> int:
        if isinstance(key, int):
            return key % self.capacity
        return (int(hashlib.sha256(str(key).encode()).hexdigest(), 16) % self.capacity)

m = MyHash(13)
print(m.hash_value(14))              # 1
print(m.hash_value(35))              # 9
print(type(m.hash_value('강아지')))   # <class 'int'>


**왜 int가 아니면 sha256을 거치나**: 문자열·리스트 같은 건 `%` 연산을 바로 못 해. 그래서 `str(key).encode()`로 바이트 문자열을 만들고 → `sha256`으로 다이제스트를 구하고 → `hexdigest()`로 16진수 문자열을 꺼내고 → `int(..., 16)`으로 정수 변환한 뒤에야 `% capacity`가 가능해져. (교재 137p)

---
### 3. 충돌 설명

- **왜 n:1인가**: 해시 함수는 무한히 많은 키를 **capacity개의 유한한 슬롯**에 밀어 넣어. 13으로 나눈 나머지는 0~12까지 13가지뿐인데 키는 무한하니까, 서로 다른 키가 같은 해시값을 갖는 건 **수학적으로 불가피**해. (비둘기집 원리)
- **18 추가 시**: `18 % 13 = 5`인데 버킷 5에 이미 값이 있음 → **충돌(collision)**. 체인법은 해시값이 같은 원소를 **연결 리스트**로 묶어서(체인 모양) 같은 버킷에 여러 개를 매달아 해결해.

---
### 4. 내장 이름 가리기 🔥

**결과**: `hash = ChainedHash(13)` 이후 `hash('abc')`를 호출하면 `TypeError: 'ChainedHash' object is not callable`.

- **원인**: `hash`는 파이썬 **내장 함수**야. 같은 이름으로 변수를 만들면 그 스코프에서 내장 함수가 **가려져(shadowing)** 버려. (우리가 계속 걸렸던 `list = []`, `sum = 0`, `max.py`와 정확히 같은 문제!)
- **영향**: 
  - 모듈 레벨(`chained_hash_test.py`)에서 덮으면 그 파일 전체에서 내장 `hash`를 못 씀.
  - 메서드 안(`hash = self.hash_value(key)`)에서 덮는 건 **지역 변수**라 함수 밖엔 영향 없어서 상대적으로 덜 위험하지만, 그 함수 안에서 내장 `hash`를 쓸 일이 생기면 똑같이 터져.
- **해결**: 다른 이름을 쓰자.
  - 테스트 파일: `hash = ChainedHash(13)` → **`table = ChainedHash(13)`** 또는 `h = ChainedHash(13)`
  - 메서드 안: `hash = self.hash_value(key)` → **`idx = self.hash_value(key)`** (해시값을 인덱스로 쓰니까 의미상으로도 더 정확)
- **대안**: 굳이 그 이름을 쓰고 싶으면 `hash_` 처럼 언더스코어를 붙이는 게 파이썬 관례(PEP 8).

> 💡 교재도 이렇게 쓰지만, 교재가 항상 최선의 스타일은 아니야. **내장 이름 목록은 `dir(__builtins__)`로 확인할 수 있어.**

---
### 5. add는 왜 맨 앞에?

```python
temp = Node(key, value, self.table[hash])  # 새 노드의 next = 기존 head
self.table[hash] = temp                    # 버킷이 새 노드를 가리키게
```

- **의미**: 새 노드를 만들면서 `next`를 **기존 head**로 설정 → 새 노드가 기존 리스트 전체를 뒤에 매달게 됨. 그 다음 버킷이 새 노드를 가리키도록 갱신. 즉 **맨 앞 삽입(head insertion)**.
- **왜 맨 앞?**: **O(1)**이니까. 맨 뒤에 추가하려면 리스트 끝까지 걸어가야 해서 **O(체인 길이)**가 돼. 어차피 `add`는 중복 검사로 이미 리스트를 다 훑지만, 맨 앞 삽입은 그 후 **포인터 조작 2번**으로 끝나.
- **dump 순서가 역순인 이유**: 나중에 넣은 게 계속 맨 앞으로 오니까 **추가 순서의 역순**(`27 → 14 → 1`)으로 보여. 이건 스택(LIFO)과 같은 원리야.

---
### 6. `pp is None` 버그

**결과**:
- `h.remove(1)` (버킷 1의 **두 번째** 노드) → 정상 동작 (`True`). `pp`가 14 노드를 가리키고 있으니 `pp.next = p.next`가 잘 됨.
- `h.remove(14)` (버킷 1의 **첫 번째** 노드) → **`AttributeError: 'NoneType' object has no attribute 'next'`**

- **원인**: 삭제 대상이 리스트의 **첫 노드(head)**면 "바로 앞의 노드"가 존재하지 않아서 `pp`가 `None`인 채로 남아. `None.next = ...`는 당연히 에러.
- **영향**: 각 버킷의 첫 노드는 절대 삭제할 수 없고 프로그램이 죽음.
- **해결**: 원본처럼 분기해야 해.
  ```python
  if pp is None:
      self.table[hash] = p.next   # 버킷이 두 번째 노드를 가리키게
  else:
      pp.next = p.next            # 앞 노드가 뒷 노드를 가리키게
  ```
- **핵심**: 연결 리스트 삭제는 **"head를 지우는 경우"**가 항상 특수 케이스야. 6일차 보초법에서 `x[0]`을 따로 받았던 것과 같은 결 — **경계 케이스를 별도로 처리**하는 패턴.

> 💡 `pp is None`에서 `==`이 아니라 `is`를 쓴 이유: `None`은 싱글턴이라 **동일성(identity)** 비교가 맞아. (3일차 `== vs is` 복습)

---
### 7. 소수 크기 실험 해설

`capacity=12`(합성수)일 땐 키들이 6의 배수라 **6과 12의 최대공약수(6)** 때문에 버킷 0과 6에만 몰려. 사용된 버킷이 2개뿐이고 체인이 길어짐.

`capacity=13`(소수)일 땐 13이 6과 서로소라서 나머지가 **고르게 퍼져**. 사용 버킷 수가 훨씬 많고 최대 체인 길이가 짧아짐.

**왜 소수인가**: 데이터가 특정 간격(예: 2의 배수, 6의 배수)으로 규칙성을 갖는 경우가 실무에 흔해. capacity가 그 간격과 **공약수를 가지면** 해시값이 몇몇 버킷에 집중돼. 소수는 대부분의 수와 서로소라 이런 패턴에 강해 → 교재 136p "해시 함수가 해시 테이블 크기보다 작거나 같은 정수를 고르게 생성해야 합니다"

---
### 8. chain_length / load_factor 정답

In [ ]:
class StatsHash(ChainedHash):
    def chain_length(self, key) -> int:
        idx = self.hash_value(key)
        p = self.table[idx]
        count = 0
        while p is not None:
            count += 1
            p = p.next
        return count

    def load_factor(self) -> float:
        total = 0
        for i in range(self.capacity):
            p = self.table[i]
            while p is not None:
                total += 1
                p = p.next
        return total / self.capacity

h = StatsHash(13)
for k in [1, 14, 27, 5]:
    h.add(k, f'v{k}')

print(h.chain_length(1))          # 3
print(h.chain_length(5))          # 1
print(h.chain_length(2))          # 0
print(round(h.load_factor(), 4))  # 0.3077


**왜**: 두 메서드 모두 `while p is not None: p = p.next`라는 **연결 리스트 순회 패턴**을 그대로 써. `search`/`add`/`remove`/`dump` 전부 이 패턴의 변주야 — 6일차 선형 검색이 배열 인덱스(`i += 1`) 대신 포인터(`p = p.next`)로 바뀐 것뿐.

**적재율(load factor)의 의미**: 이 값이 1을 넘어가기 시작하면 평균 체인 길이가 1보다 길어져서 O(1)이 깨져. 실무 해시맵(파이썬 `dict`, 자바 `HashMap`)은 적재율이 임계값(보통 0.75)을 넘으면 테이블 크기를 2배로 늘리고 전부 다시 해싱(**rehashing**)해.

---
### 9. 어노테이션 & 트레이드오프

**(a) `-> int`인데 왜 에러가 안 나나**:
파이썬 어노테이션은 **강제되지 않는 힌트**야 (4일차). 인터프리터는 `-> int`를 그냥 `__annotations__` 딕셔너리에 메타데이터로 저장할 뿐, 실제 반환값이 int인지 **검사하지 않아**. `__init__`은 어차피 `None`을 반환하는데도 조용히 잘 돌아가는 이유.

**그럼 왜 쓰나**: (1) 사람이 읽는 문서 역할, (2) IDE 자동완성·경고, (3) `mypy` 같은 정적 타입 검사기가 미리 잡아줌. 즉 **런타임이 아니라 개발 시점**에 값어치가 있어. 그러니 틀린 어노테이션은 "동작은 하지만 거짓말하는 문서"라 오히려 해로워 → `-> None`으로 고치는 게 맞아.

**(b) 트레이드오프**:
- **capacity를 무한히 크게**: 충돌이 거의 없어져서 검색·추가·삭제가 진짜 O(1)에 수렴 (**시간 이득**). 하지만 대부분 `None`인 빈 버킷이 메모리를 잡아먹음 (**공간 손해**). 원소 10개에 테이블 100만 칸이면 낭비.
- **capacity = 1**: 모든 키의 해시값이 `key % 1 == 0` → 전부 버킷 0에 몰림 → **하나의 긴 연결 리스트**가 됨. 즉 그냥 **연결 리스트**야. 검색 복잡도는 **O(n)** — 해시법의 이점이 완전히 사라지고 6일차 선형 검색으로 퇴화.

이게 "해시 테이블 크기를 어떻게 잡느냐"가 해시법의 핵심인 이유야. 너무 작으면 O(n)으로 퇴화, 너무 크면 메모리 낭비.

---

## 📌 핵심 3줄 요약

1. **해시법**은 "저장할 위치 = 인덱스"를 `key % capacity` 같은 계산으로 바로 구해서, 정렬 배열의 O(n) 원소 이동 비용 없이 **검색·추가·삭제를 모두 O(1)에 가깝게** 만든다.
2. 키:해시값이 **n:1**이라 **충돌**은 필연 → **체인법**은 같은 해시값 원소를 **연결 리스트**로 묶어 해결하고, `search`/`add`/`remove` 전부 "해시값 → 버킷 → 리스트 선형 검색"이라는 **같은 패턴**이다.
3. capacity가 작으면 체인이 길어져 **O(n)으로 퇴화**, 크면 메모리 낭비 → **시간·공간 트레이드오프**. 그래서 크기는 **소수**를 선호한다.

## 🗂️ 스터디 진행 가이드
- 🟢 (1~3번): 전원 필수. 특히 1번 해시값 손계산은 해시법 이해의 출발점
- 🟡 (4~6번): 팀 목표선.
  - **4번 shadowing은 반드시** — 우리가 `list`, `sum`, `max`로 계속 걸렸던 바로 그 패턴이 교재 코드에도 들어있음
  - **6번 `pp is None`도 반드시** — 연결 리스트 삭제의 head 특수 케이스는 8장 연결 리스트에서 또 나옴
- 🔴 (7~9번): 도전. 8번 load_factor는 실무 해시맵(dict, HashMap) 이해로 이어짐
- **금요일 코딩테스트 범위**: 🟢🟡 (1~6번)

## 🔗 지금까지 배운 검색 3형제 총정리

| | 선형 검색 | 이진 검색 | 해시법(체인법) |
|---|---|---|---|
| **전제** | 없음 | **정렬** 필수 | 없음 |
| **검색** | O(n) | O(log n) | **O(1)** (충돌 없을 때) |
| **추가** | O(1) | O(n) (원소 이동) | **O(1)** |
| **삭제** | O(n) | O(n) (원소 이동) | **O(1)** |
| **추가 공간** | 없음 | 없음 | 테이블 + 노드 |
| **일차** | 6일차 | 7일차 | 8일차 |

> 교재 110p의 결론: **"용도, 목적, 실행 속도, 자료구조 등 여러 사항을 고려해서 선택해야 합니다."**
